# 23CSE301 Review 2 — Clustering
## Customer Personality Analysis

**Goal:** run K-Means and Agglomerative Hierarchical Clustering, report all required clustering metrics, and create Elbow, dendrogram, PCA, and t-SNE visualizations.

### Before running
1. Download [Customer Personality Analysis](https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis).
2. Extract the dataset and put its CSV inside a `data` folder beside this notebook.
3. Run every cell from top to bottom.

The common Kaggle file is named `marketing_campaign.csv`, but this notebook searches for CSV files. If multiple files are found, set `CSV_PATH` to the right one.

**Important:** clustering is unsupervised. Do not use customer response/label columns to fit clusters. This notebook excludes common outcome columns and ID columns from clustering features.

## 1. Import packages

If packages are missing, uncomment `%pip install ...` below.

In [ ]:
# Uncomment only if imports fail:
# %pip install pandas numpy matplotlib seaborn scikit-learn scipy

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
sns.set_theme()
print("Imports completed.")

## 2. Load the dataset

In [ ]:
DATA_FOLDER = Path("data")
CSV_PATH = None  # Example: Path("data/marketing_campaign.csv")

if CSV_PATH is not None:
    csv_path = Path(CSV_PATH)
else:
    candidates = []
    for root in [DATA_FOLDER, Path(".")]:
        if root.exists():
            candidates.extend([p for p in root.glob("*.csv") if p not in candidates])
    candidates = sorted(set(candidates), key=lambda p: ("marketing_campaign" not in p.name.lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError(
            "No CSV found. Create a 'data' folder beside this notebook and put the customer dataset CSV inside it."
        )
    print("CSV files found:")
    for i, p in enumerate(candidates):
        print(f"{i}: {p}")
    csv_path = candidates[0]

# The common marketing_campaign.csv is tab-separated; try comma first, then tab if it loads as one column.
df = pd.read_csv(csv_path)
if df.shape[1] == 1:
    df = pd.read_csv(csv_path, sep="\t")

print("Loaded:", csv_path)
print("Shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

## 3. Audit and clean the data

We inspect missing values, duplicates, and data types. The standard customer personality dataset sometimes has missing `Income` values and a date column named `Dt_Customer`.

In [ ]:
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=False)
}).sort_values("missing_percent", ascending=False))

print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Convert the customer registration date into numeric age-in-days, if available.
date_columns = [col for col in df.columns if col.lower() in {"dt_customer", "date_customer", "customer_date"}]
for col in date_columns:
    dates = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
    if dates.notna().any():
        reference_date = dates.max()
        df[col + "_days_since_customer"] = (reference_date - dates).dt.days
    df = df.drop(columns=[col])

display(df.head())

## 4. Choose features for clustering

- Remove identifiers such as `ID`.
- Remove response/outcome columns (for example, `Response` and `Complain`) so the clustering is based on customer characteristics and behaviour rather than an outcome.
- Convert categorical variables using one-hot encoding.
- Impute missing values and scale features.
- Do not use a true ground-truth label during clustering. This dataset generally does not provide a ground-truth cluster label.

In [ ]:
# Exclude known IDs and outcome/label-like columns if they exist.
exclude_exact = {
    "id", "customer_id", "customerid",
    "response", "complain", "target", "label", "cluster"
}
exclude_columns = [
    col for col in df.columns
    if col.lower().replace(" ", "").replace("_", "") in
       {name.replace("_", "") for name in exclude_exact}
]
print("Excluded columns:", exclude_columns)

X = df.drop(columns=exclude_columns, errors="ignore").copy()

# Remove columns that are entirely missing or constant.
empty_cols = X.columns[X.isna().all()].tolist()
constant_cols = [col for col in X.columns if X[col].nunique(dropna=False) <= 1]
drop_cols = sorted(set(empty_cols + constant_cols))
if drop_cols:
    print("Dropping empty/constant columns:", drop_cols)
    X = X.drop(columns=drop_cols)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [col for col in X.columns if col not in numeric_features]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
], verbose_feature_names_out=False)

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()
X_processed = np.asarray(X_processed, dtype=float)

print("Original feature rows:", X.shape[0])
print("Processed matrix shape:", X_processed.shape)
print("Numeric columns:", len(numeric_features))
print("Categorical columns:", len(categorical_features))

## 5. Elbow curve for K-Means

We test multiple values of `k`. Look for a bend in the curve where adding more clusters gives diminishing improvement. The elbow is a guide, not an automatic proof of the best `k`.

In [ ]:
# Use a modest range. You can extend this after the first run.
max_k = min(10, X_processed.shape[0] - 1)
if max_k < 2:
    raise ValueError("Need at least 3 rows for clustering.")
k_values = list(range(2, max_k + 1))
inertias = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_processed)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, marker="o")
plt.title("Elbow Curve for K-Means")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.xticks(k_values)
plt.tight_layout()
plt.show()

## 6. Dendrogram for Agglomerative Hierarchical Clustering

A dendrogram is computationally expensive for very large datasets. We plot a random sample of up to 500 rows so the plot remains readable. The clustering metrics below still use the full processed dataset.

In [ ]:
sample_size = min(500, X_processed.shape[0])
rng = np.random.default_rng(RANDOM_STATE)
sample_indices = rng.choice(X_processed.shape[0], size=sample_size, replace=False)
X_dendrogram = X_processed[sample_indices]

# Ward linkage is appropriate for Euclidean distance on scaled numeric feature vectors.
Z = linkage(X_dendrogram, method="ward")
plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=45, leaf_font_size=10)
plt.title("Agglomerative Clustering Dendrogram (sample; last 30 merged groups)")
plt.xlabel("Sampled observations / merged groups")
plt.ylabel("Ward distance")
plt.tight_layout()
plt.show()

## 7. Fit both required clustering algorithms

The guidelines specifically require K-Means and Agglomerative Hierarchical Clustering. Use the Elbow curve and dendrogram to inform your choice of `N_CLUSTERS`. Change it if your interpretation of the plots supports another value.

In [ ]:
N_CLUSTERS = 4  # Change this after reviewing the Elbow curve and dendrogram.

if N_CLUSTERS >= X_processed.shape[0]:
    raise ValueError("N_CLUSTERS must be smaller than the number of rows.")

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_processed)

agg = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="ward")
agg_labels = agg.fit_predict(X_processed)

print("K-Means cluster sizes:")
display(pd.Series(kmeans_labels).value_counts().sort_index().rename("count").to_frame())
print("Agglomerative cluster sizes:")
display(pd.Series(agg_labels).value_counts().sort_index().rename("count").to_frame())

## 8. Evaluate clusters with all required metrics

Required metrics:
- **Silhouette Score:** higher is generally better; measures separation and cohesion.
- **Davies–Bouldin Index:** lower is generally better.
- **Calinski–Harabasz Index:** higher is generally better.

These metrics require at least two clusters and fewer clusters than observations.

In [ ]:
def clustering_metrics(X_array, labels, model_name):
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2 or len(unique_labels) >= len(labels):
        return {
            "Algorithm": model_name,
            "Silhouette Score": np.nan,
            "Davies-Bouldin Index": np.nan,
            "Calinski-Harabasz Index": np.nan
        }
    return {
        "Algorithm": model_name,
        "Silhouette Score": silhouette_score(X_array, labels),
        "Davies-Bouldin Index": davies_bouldin_score(X_array, labels),
        "Calinski-Harabasz Index": calinski_harabasz_score(X_array, labels)
    }

clustering_results = pd.DataFrame([
    clustering_metrics(X_processed, kmeans_labels, "K-Means"),
    clustering_metrics(X_processed, agg_labels, "Agglomerative Hierarchical")
])
display(clustering_results.round(4))

## 9. PCA 2D visualizations for both algorithms

PCA reduces the processed features to two principal components so we can visualize the clusters. The PCA plot is for visualization; the clustering itself was performed on the preprocessed feature matrix.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_processed)
print("Variance explained by first two PCs:", round(pca.explained_variance_ratio_.sum(), 4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, s=12, alpha=0.7, cmap="tab10")
axes[0].set_title("K-Means clusters — PCA 2D")
axes[0].set_xlabel("Principal Component 1")
axes[0].set_ylabel("Principal Component 2")

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=agg_labels, s=12, alpha=0.7, cmap="tab10")
axes[1].set_title("Agglomerative clusters — PCA 2D")
axes[1].set_xlabel("Principal Component 1")
axes[1].set_ylabel("Principal Component 2")

plt.tight_layout()
plt.show()

## 10. t-SNE visualization (at least one algorithm)

t-SNE is computationally heavier than PCA. For speed, this cell uses up to 2,000 randomly selected observations. The t-SNE plot is a visualization only; it does not train the clustering models.

In [ ]:
tsne_size = min(2000, X_processed.shape[0])
tsne_indices = rng.choice(X_processed.shape[0], size=tsne_size, replace=False)
X_tsne_input = X_processed[tsne_indices]
labels_tsne = kmeans_labels[tsne_indices]

perplexity_value = min(30, max(5, (tsne_size - 1) // 3))
perplexity_value = min(perplexity_value, tsne_size - 1)

tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=perplexity_value,
            init="pca", learning_rate="auto")
X_tsne = tsne.fit_transform(X_tsne_input)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_tsne, s=12, alpha=0.7, cmap="tab10")
plt.title("K-Means clusters — t-SNE (sample)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.tight_layout()
plt.show()

## 11. Profile the clusters

Cluster profiles help describe how groups differ. These summaries are descriptive, not proof that a cluster is inherently good or bad. Numeric summaries are calculated from the original numeric columns, with the assigned cluster label added.

In [ ]:
profile_df = df.loc[X.index].copy()
profile_df["KMeans_Cluster"] = kmeans_labels
profile_df["Agglomerative_Cluster"] = agg_labels

original_numeric = profile_df.select_dtypes(include=np.number).columns.tolist()
original_numeric = [col for col in original_numeric if col not in {"KMeans_Cluster", "Agglomerative_Cluster"}]

print("K-Means numeric profile:")
display(profile_df.groupby("KMeans_Cluster")[original_numeric].mean().round(2))

print("Agglomerative numeric profile:")
display(profile_df.groupby("Agglomerative_Cluster")[original_numeric].mean().round(2))

## 12. Save labels and metric table

In [ ]:
output = df.loc[X.index].copy()
output["KMeans_Cluster"] = kmeans_labels
output["Agglomerative_Cluster"] = agg_labels
output.to_csv("customer_personality_cluster_assignments.csv", index=False)
clustering_results.to_csv("clustering_metrics.csv", index=False)

print("Saved:")
print("- customer_personality_cluster_assignments.csv")
print("- clustering_metrics.csv")

## 13. Review 2 checklist

- [ ] K-Means fitted and cluster labels obtained.
- [ ] Agglomerative Hierarchical Clustering fitted and cluster labels obtained.
- [ ] Silhouette, Davies–Bouldin, and Calinski–Harabasz metrics reported for both.
- [ ] K-Means Elbow curve displayed.
- [ ] Dendrogram displayed.
- [ ] PCA 2D cluster visualization shown for each algorithm.
- [ ] t-SNE visualization shown for at least one algorithm.
- [ ] Notebook runs top-to-bottom without errors.
- [ ] Add your own Markdown interpretation of the plots and metrics. The course guidelines require original analysis and interpretation.